# Machine Learning

Hier wird nun das beste Modell genommen und auf unbekannte Daten getestet. Es wird hierbei davon ausgegangen, dass die Daten im gleichen Format wie die Train.npz Daten aus dem Ordner Model_data zur verfügung gestellt werden.

## Bibliotheken

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

## 1. Daten einlesen

In [3]:
data_path = Path("../../Model_data")

data = np.load(data_path / "test.npz")
X_test_raw = data["X"]
y_test_raw = data["y"]

print(f"X_test_raw.shape: {X_test_raw.shape},\ny_test_raw.shape: {y_test_raw.shape}")
print(f"X_test_raw.dtype: {X_test_raw.dtype},\ny_test_raw.dtype: {y_test_raw.dtype}")


X_test_raw.shape: (10273, 100, 13),
y_test_raw.shape: (10273,)
X_test_raw.dtype: float64,
y_test_raw.dtype: <U9


## 2. Model Artefakte einlesen

In [4]:
model_path = Path("../../Model_data/ML_Daten/final_model.joblib")
artifact = joblib.load(model_path)
artifact.keys(), type(artifact["model"]), type(artifact["scaler"]), type(artifact["label_encoder"])

(dict_keys(['model', 'scaler', 'label_encoder', 'class_names', 'best_params']),
 sklearn.ensemble._forest.RandomForestClassifier,
 sklearn.preprocessing._data.StandardScaler,
 sklearn.preprocessing._label.LabelEncoder)

In [5]:
def _extract_channel_features(channel: np.ndarray) -> list:
    mean = np.mean(channel)
    std = np.std(channel)
    min_val = np.min(channel)
    max_val = np.max(channel)
    median = np.median(channel)
    iqr = np.percentile(channel, 75) - np.percentile(channel, 25)
    rms = np.sqrt(np.mean(channel ** 2))
    signal_range = max_val - min_val
    diffs = np.diff(channel)
    mean_abs_diff = np.mean(np.abs(diffs))
    zero_crossings = np.sum(np.diff(np.sign(channel - mean)) != 0)
    t = np.arange(len(channel), dtype=np.float64)
    slope = np.polyfit(t, channel, 1)[0] if std > 1e-10 else 0.0
    return [mean, std, min_val, max_val, median, iqr,
            rms, signal_range, mean_abs_diff, zero_crossings, slope]
def _extract_window_features(window: np.ndarray) -> np.ndarray:
    feats = []
    for ch_idx in range(window.shape[1]):
        feats.extend(_extract_channel_features(window[:, ch_idx]))
    acc_mag = np.sqrt(window[:, 0]**2 + window[:, 1]**2 + window[:, 2]**2)
    feats.extend(_extract_channel_features(acc_mag))
    gyr_mag = np.sqrt(window[:, 3]**2 + window[:, 4]**2 + window[:, 5]**2)
    feats.extend(_extract_channel_features(gyr_mag))
    return np.array(feats, dtype=np.float64)
def build_feature_matrix(X_windows: np.ndarray) -> np.ndarray:
    n_samples = X_windows.shape[0]
    sample = _extract_window_features(X_windows[0])
    X_feat = np.empty((n_samples, len(sample)), dtype=np.float64)
    X_feat[0] = sample
    for i in range(1, n_samples):
        X_feat[i] = _extract_window_features(X_windows[i])
    return X_feat

In [6]:
model = artifact["model"]
scaler = artifact["scaler"]
le = artifact["label_encoder"]
X5 = X_test_raw[:5]
y5_true = y_test_raw[:5]
X5_feat = build_feature_matrix(X5)
X5_feat.shape

(5, 165)

## 3. Skalieren, vorhersagen

In [7]:
X5_scaled = scaler.transform(X5_feat)
y5_pred_enc = model.predict(X5_scaled)
y5_pred = le.inverse_transform(y5_pred_enc)
list(zip(y5_true.tolist(), y5_pred.tolist()))

[('Roundkick', 'Roundkick'),
 ('Roundkick', 'Roundkick'),
 ('Roundkick', 'Roundkick'),
 ('Roundkick', 'Roundkick'),
 ('Treppe', 'Treppe')]

## 6. Überprüfbarkeit herstellen

In [8]:
N = 500  # später z.B. 4112 fürs ganze Testset
Xn = X_test_raw[:N]
yn_true = y_test_raw[:N]
Xn_feat = build_feature_matrix(Xn)
Xn_scaled = scaler.transform(Xn_feat)
yn_pred = le.inverse_transform(model.predict(Xn_scaled))
acc = (yn_pred == yn_true).mean()
acc

np.float64(0.96)

In [9]:

pd.crosstab(pd.Series(yn_true, name="true"),
            pd.Series(yn_pred, name="pred"),
            normalize="index").round(2)

pred,Auto,Laufen,Roundkick,Treppe,Velo,Zug
true,,,,,,
Auto,1.0,0.00,0.0,0.00,0.00,0.0
Laufen,0.0,0.96,0.0,0.01,0.03,0.0
Roundkick,0.0,0.00,1.0,0.00,0.00,0.0
Treppe,0.0,0.21,0.0,0.76,0.03,0.0
Zug,0.0,0.00,0.0,0.00,0.00,1.0


## Majority Vote für eine Session

In [10]:
start = 0
K = 200
block_true = y_test_raw[start:start+K]

# Features berechnen und skalieren
X_block = X_test_raw[start:start+K]
X_block_feat = build_feature_matrix(X_block)
X_block_scaled = scaler.transform(X_block_feat)

# Probability statt Majority Vote
proba = model.predict_proba(X_block_scaled)
mean_proba = proba.mean(axis=0)
maj_pred = le.inverse_transform([mean_proba.argmax()])[0]
confidence = mean_proba.max()
maj_true = pd.Series(block_true).value_counts().idxmax()

print(f"Wahr:      {maj_true}")
print(f"Vorhergesagt: {maj_pred} ({confidence:.0%} Konfidenz)")

Wahr:      Laufen
Vorhergesagt: Laufen (78% Konfidenz)


## 7. Predict Funktion für das .py später

In [11]:
def predict_windows_block(X_windows, artifact):
    model = artifact["model"]
    scaler = artifact["scaler"]
    le = artifact["label_encoder"]

    X_feat = build_feature_matrix(X_windows)
    X_scaled = scaler.transform(X_feat)
    y_pred = le.inverse_transform(model.predict(X_scaled))

    vc = pd.Series(y_pred).value_counts()
    majority = vc.idxmax()
    distribution = (vc / vc.sum()).round(3)

    return {
        "predictions": y_pred,
        "majority_vote": majority,
        "distribution": distribution,
        "n_windows": len(y_pred),
    }

In [12]:
out = predict_windows_block(X_test_raw[start:start+K], artifact)
out["majority_vote"], out["distribution"]

('Laufen',
 Laufen       0.805
 Treppe       0.125
 Velo         0.050
 Roundkick    0.020
 Name: count, dtype: float64)

# Diese Zelle kann später in ein .py kopiert werden
Vorausgesetzt es bleibt dabei was zu oberst steht, dass die Daten das selbe Format wie die train.npz haben

In [13]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# ---- Konfiguration ----
#npz_path = Path("../../Model_data/test.npz")  # später: neue Datei
npz_path = Path("../../Model_data/test_silas.npz")  # später: neue Datei
model_path = Path("../../Model_data/ML_Daten/final_model.joblib")

# ---- Feature Engineering (muss identisch zu Training sein) ----
def _extract_channel_features(channel: np.ndarray) -> list:
    mean = np.mean(channel)
    std = np.std(channel)
    min_val = np.min(channel)
    max_val = np.max(channel)
    median = np.median(channel)
    iqr = np.percentile(channel, 75) - np.percentile(channel, 25)

    rms = np.sqrt(np.mean(channel ** 2))
    signal_range = max_val - min_val
    diffs = np.diff(channel)
    mean_abs_diff = np.mean(np.abs(diffs))
    zero_crossings = np.sum(np.diff(np.sign(channel - mean)) != 0)

    t = np.arange(len(channel), dtype=np.float64)
    slope = np.polyfit(t, channel, 1)[0] if std > 1e-10 else 0.0

    return [mean, std, min_val, max_val, median, iqr,
            rms, signal_range, mean_abs_diff, zero_crossings, slope]

def _extract_window_features(window: np.ndarray) -> np.ndarray:
    feats = []
    for ch_idx in range(window.shape[1]):
        feats.extend(_extract_channel_features(window[:, ch_idx]))

    acc_mag = np.sqrt(window[:, 0]**2 + window[:, 1]**2 + window[:, 2]**2)
    feats.extend(_extract_channel_features(acc_mag))

    gyr_mag = np.sqrt(window[:, 3]**2 + window[:, 4]**2 + window[:, 5]**2)
    feats.extend(_extract_channel_features(gyr_mag))

    return np.array(feats, dtype=np.float64)

def build_feature_matrix(X_windows: np.ndarray) -> np.ndarray:
    n_samples = X_windows.shape[0]
    sample = _extract_window_features(X_windows[0])
    X_feat = np.empty((n_samples, len(sample)), dtype=np.float64)
    X_feat[0] = sample
    for i in range(1, n_samples):
        X_feat[i] = _extract_window_features(X_windows[i])
    return X_feat

# ---- Laden ----
data = np.load(npz_path)
X = data["X"]  # erwartet: (n_windows, 100, 13)

artifact = joblib.load(model_path)
model = artifact["model"]
scaler = artifact["scaler"]
le = artifact["label_encoder"]

# ---- Predict ----
X_feat = build_feature_matrix(X)
X_scaled = scaler.transform(X_feat)

# Probability statt Majority Vote
proba = model.predict_proba(X_scaled)
mean_proba = proba.mean(axis=0)
majority = le.inverse_transform([mean_proba.argmax()])[0]
confidence = mean_proba.max()

print(f"{majority} ({confidence:.0%})")

Laufen (53%)


In [ ]:
print((vc / vc.sum()).round(3))_

NameError: name 'vc' is not defined